# Looped Block-ELL Transformer — Scaling Law Explorer

**Goal**: Study how width, depth, and pruning density interact in Parcae-style looped transformers.

Architecture:
- **Prelude** (non-looped) → **Input norm** → **Core** (looped T times via `lax.scan` with diagonal injection) → **Coda** (non-looped)
- Diagonal injection: `h_new = decay * h + dt * e`, guaranteed stable by construction
- Poisson depth sampling per sequence, BPTT truncation, sequences frozen via `jnp.where`
- MLP pruning via CMS (gradient Frobenius norm tile scoring), masked dense weights

**Change Cell 2 to run a sweep.** Everything else is automatic.

> ⚡ Runtime: TPU v2-8 or v3-8 recommended. Go to `Runtime → Change runtime type → TPU`.

## Cell 1 — Setup & Install

Installs JAX with TPU support, Flax, Optax, and helpers. Run once per session.

In [ ]:
# ─── Install ──────────────────────────────────────────────────────────────────
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("jax[tpu]", "-f", "https://storage.googleapis.com/jax-releases/libtpu_releases.html")
_pip("flax", "optax", "wandb", "datasets<3.0", "transformers", "tqdm")

import os
import jax
import jax.numpy as jnp

# ─── wandb login (uses Colab secret WANDB_API_KEY) ───────────────────────────
import wandb
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except (ImportError, Exception):
    pass
wandb.login()

# ─── Mount Google Drive for persistent results ───────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_DIR = "/content/drive/MyDrive/looped-blockell-scaling"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Results will be saved to: {DRIVE_DIR}/")

print(f"JAX {jax.__version__}")
print(f"Devices ({len(jax.devices())}): {jax.devices()}")
assert len(jax.devices()) >= 1, "No TPU devices found. Go to Runtime → Change runtime type → TPU."
assert jax.devices()[0].platform == "tpu", f"Expected TPU, got {jax.devices()[0].platform}. Change runtime type."

## Cell 2 — Grid Search Config

20-point grid over (d_model, n_core) — all other hyperparameters fixed.
Results are saved incrementally to `scaling_results.json` after each run.

Grid axes:
- **Width**: d_model ∈ {256, 384, 512, 768, 1024}
- **Core depth**: n_core ∈ {2, 4, 6, 8} (looped core layers, T=4 iterations each)

Effective depth = N_PRELUDE + n_core × MEAN_DEPTH + N_CODA = 1 + n_core×4 + 1

**Compute budget**: normalized to 20 tok/param for the reference model (d=1024, c=4, ~126M params).
77k steps × batch=32 × seq=1024 = 2.52B tokens consumed.
840M unique tokens × 3 epochs = 2.52B. Prune schedule completes at step 60k with 17k recovery steps.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  GRID SEARCH CONFIG — (d_model, n_heads, d_ff, n_core)
# ═══════════════════════════════════════════════════════════════════════════════

GRID = [
    (256,   4,  1024, 2),
    (256,   4,  1024, 4),
    (256,   4,  1024, 6),
    (256,   4,  1024, 8),
    (384,   6,  1536, 2),
    (384,   6,  1536, 4),
    (384,   6,  1536, 6),
    (384,   6,  1536, 8),
    (512,   8,  2048, 2),
    (512,   8,  2048, 4),
    (512,   8,  2048, 6),
    (512,   8,  2048, 8),
    (768,  12,  3072, 2),
    (768,  12,  3072, 4),
    (768,  12,  3072, 6),
    (768,  12,  3072, 8),
    (1024, 16,  4096, 2),
    (1024, 16,  4096, 4),
    (1024, 16,  4096, 6),
    (1024, 16,  4096, 8),
]

# ── Fixed across all configs ──────────────────────────────────────────────────
N_PRELUDE = 1
N_CODA    = 1

MEAN_DEPTH  = 4
MAX_DEPTH   = 4
BPTT_DEPTH  = 4
USE_POISSON = True

TILE_SIZE      = 16
PRUNE_FRACTION = 0.10
PRUNE_INTERVAL = 4600
PRUNE_START    = 5000
PRUNE_END      = 60000

# Normalized to 20 tok/param for d=1024/c=4 (~126M params)
# 126M × 20 / 32768 ≈ 77k steps, 3 epochs over 840M unique tokens
TOTAL_STEPS  = 77000
BATCH_SIZE   = 32
LR           = 6e-4
WARMUP_STEPS = 2000
GRAD_CLIP    = 1.0
MAX_SEQ_LEN  = 1024
VOCAB_SIZE   = 49152
WEIGHT_DECAY = 0.1
SEED         = 42

WANDB_PROJECT = "looped-blockell-scaling"
EVAL_INTERVAL = 2500
LOG_INTERVAL  = 50
SCORE_INTERVAL = 10

# ── Derived ───────────────────────────────────────────────────────────────────
N_PRUNE_ROUNDS = max(0, (PRUNE_END - PRUNE_START) // PRUNE_INTERVAL)
FINAL_DENSITY  = (1.0 - PRUNE_FRACTION) ** N_PRUNE_ROUNDS if N_PRUNE_ROUNDS > 0 else 1.0
TOKENS_PER_STEP = BATCH_SIZE * MAX_SEQ_LEN
TOTAL_TOKENS    = TOTAL_STEPS * TOKENS_PER_STEP

print(f"Grid size    : {len(GRID)} configs")
print(f"Fixed depth  : mean_T={MEAN_DEPTH}, max_T={MAX_DEPTH}, bptt={BPTT_DEPTH}")
print(f"Prune        : {N_PRUNE_ROUNDS} rounds @ {PRUNE_FRACTION:.0%}/round → {FINAL_DENSITY:.1%} final density")
print(f"Training     : {TOTAL_STEPS} steps, batch={BATCH_SIZE}, lr={LR}")
print(f"Tokens       : {TOTAL_TOKENS/1e9:.2f}B consumed, 3 epochs over 840M unique")
print(f"Recovery     : {TOTAL_STEPS - PRUNE_END} steps post-prune")
print()

def estimate_params(d_model, n_heads, d_ff, n_core):
    vocab    = VOCAB_SIZE * d_model
    attn     = d_model * (3 * d_model + d_model)
    mlp      = d_model * d_ff + d_ff + d_ff * d_model + d_model
    block    = attn + mlp
    n_blocks = N_PRELUDE + n_core + N_CODA
    norms    = n_blocks * 2 * d_model + d_model
    inj      = d_model * 2
    return vocab + n_blocks * block + norms + inj

print(f"{'d_model':>8} {'n_core':>7} {'eff_depth':>10} {'~params':>10} {'tok/param':>10}")
print("─" * 50)
for d_model, n_heads, d_ff, n_core in GRID:
    eff = N_PRELUDE + n_core * MEAN_DEPTH + N_CODA
    p   = estimate_params(d_model, n_heads, d_ff, n_core)
    tp  = TOTAL_TOKENS / p
    print(f"{d_model:>8} {n_core:>7} {eff:>10} {p/1e6:>9.1f}M {tp:>9.0f}×")

## Cell 3 — Streaming Data Loader

Sequential token consumption from Dolma v1.7 via HuggingFace streaming.
Background thread tokenizes ahead while TPU trains — CPU is otherwise idle at ~2%.

- **840M unique tokens** loaded into train buffer (3 epochs at 77k steps × batch=32)
- **20M tokens** held out for eval (separate slice, never trained on)
- Sequential consumption: each `get_batch()` advances a pointer, no random sampling
- After exhausting the buffer, wraps around for next epoch

In [ ]:
import numpy as np
import time as _time
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm

print("Loading StarCoder2 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("bigcode/starcoder2-7b", trust_remote_code=True)
EOT = tokenizer.eos_token_id

TRAIN_TOKENS = 840_000_000
VAL_TOKENS   = 20_000_000

TRAIN_CACHE = os.path.join(DRIVE_DIR, "train_buf.npy")
VAL_CACHE   = os.path.join(DRIVE_DIR, "val_buf.npy")

if os.path.exists(TRAIN_CACHE) and os.path.exists(VAL_CACHE):
    print("Loading cached token buffers from Drive...")
    t0 = _time.time()
    train_buf = np.load(TRAIN_CACHE)
    val_buf   = np.load(VAL_CACHE)
    print(f"Loaded in {_time.time() - t0:.1f}s")
else:
    def build_token_buffer(target_tokens: int) -> np.ndarray:
        ds = load_dataset("allenai/dolma", split="train", streaming=True, trust_remote_code=True)
        buf = []
        with tqdm(total=target_tokens, unit="tok", desc="tokenise") as pbar:
            for sample in ds:
                text = sample.get("text", "")
                if not text:
                    continue
                ids = tokenizer.encode(text, add_special_tokens=False)
                ids.append(EOT)
                buf.extend(ids)
                pbar.update(len(ids))
                if len(buf) >= target_tokens:
                    break
        return np.array(buf[:target_tokens], dtype=np.int32)

    t0 = _time.time()
    total_needed = TRAIN_TOKENS + VAL_TOKENS
    print(f"Streaming {total_needed/1e6:.0f}M tokens from Dolma (one-time, cached to Drive)...")
    all_buf = build_token_buffer(total_needed)
    train_buf = all_buf[:TRAIN_TOKENS]
    val_buf   = all_buf[TRAIN_TOKENS:]
    del all_buf

    print(f"Caching to Drive...")
    np.save(TRAIN_CACHE, train_buf)
    np.save(VAL_CACHE, val_buf)
    print(f"Data loaded and cached in {(_time.time() - t0)/60:.1f} min")

print(f"Train: {len(train_buf):,} tokens ({len(train_buf)/1e6:.0f}M)")
print(f"Val  : {len(val_buf):,} tokens ({len(val_buf)/1e6:.0f}M)")
print(f"Epochs: {TOTAL_TOKENS / len(train_buf):.1f}x over train buffer")

class SequentialSampler:
    def __init__(self, buf: np.ndarray, batch_size: int, seq_len: int):
        self.buf = buf
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.pos = 0
        self.epoch = 0

    def get_batch(self):
        seqs = []
        for _ in range(self.batch_size):
            end = self.pos + self.seq_len + 1
            if end > len(self.buf):
                self.pos = 0
                self.epoch += 1
                end = self.seq_len + 1
            seqs.append(self.buf[self.pos:end])
            self.pos += self.seq_len
        arr = np.stack(seqs).astype(np.int32)
        x = jnp.asarray(arr[:, :-1])
        y = jnp.asarray(arr[:, 1:])
        return x, y

train_sampler = SequentialSampler(train_buf, BATCH_SIZE, MAX_SEQ_LEN)

val_rng = np.random.default_rng(SEED + 1)
def get_val_batch():
    starts = val_rng.integers(0, len(val_buf) - MAX_SEQ_LEN - 1, size=BATCH_SIZE)
    seqs   = np.stack([val_buf[s:s + MAX_SEQ_LEN + 1].astype(np.int32) for s in starts])
    x = jnp.asarray(seqs[:, :-1])
    y = jnp.asarray(seqs[:, 1:])
    return x, y

x, y = train_sampler.get_batch()
print(f"Batch shape: x={x.shape}, y={y.shape}, dtype={x.dtype}")

## Cell 4 — Model Implementation

All model code in one cell. Key components:

- **RMSNorm** — no bias, fp32 accumulation
- **RoPE** — precomputed frequency table, applied per QK head
- **MultiHeadAttention** — causal, no bias projections
- **MLPBlock** — dense weights with an external tile mask (mask applied via reshape-multiply)
- **DiagonalInjection** — `decay = exp(softplus(log_dt) * -exp(log_A))`, guaranteed ∈ (0,1)
- **TransformerBlock** — pre-norm residual
- **LoopedTransformer** — `lax.scan` over core blocks, `jax.checkpoint` for O(1) activation memory

**MLP pruning strategy**: weights stay dense, a boolean tile mask `[R, C]` (where R=d_out//B, C=d_in//B)
is applied by zeroing tile-shaped regions before the matmul. This is simpler than full Block-ELL
sparse indexing and produces identical scaling-law results since the gradient structure is the same.

In [ ]:
import math
from functools import partial
from typing import Optional, NamedTuple
from dataclasses import dataclass

import jax
import jax.numpy as jnp
import flax.linen as nn
import optax


# ─── RMSNorm ──────────────────────────────────────────────────────────────────

class RMSNorm(nn.Module):
    eps: float = 1e-6

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        scale = self.param("scale", nn.initializers.ones, (x.shape[-1],))
        x32 = x.astype(jnp.float32)
        rms  = jnp.sqrt((x32 ** 2).mean(axis=-1, keepdims=True) + self.eps)
        return ((x32 / rms) * scale).astype(x.dtype)


# ─── RoPE ─────────────────────────────────────────────────────────────────────

def precompute_rope_freqs(head_dim: int, max_seq_len: int, theta: float = 10000.0) -> jnp.ndarray:
    half     = head_dim // 2
    inv_freq = 1.0 / (theta ** (jnp.arange(half, dtype=jnp.float32) / half))
    t        = jnp.arange(max_seq_len, dtype=jnp.float32)
    freqs    = jnp.outer(t, inv_freq)
    return jnp.stack([jnp.cos(freqs), jnp.sin(freqs)], axis=-1)


def apply_rope(x: jnp.ndarray, freqs: jnp.ndarray) -> jnp.ndarray:
    B, H, S, D = x.shape
    half = D // 2
    x1, x2 = x[..., :half], x[..., half:]
    cos = freqs[:S, :, 0][None, None]
    sin = freqs[:S, :, 1][None, None]
    return jnp.concatenate([x1 * cos - x2 * sin, x1 * sin + x2 * cos], axis=-1)


# ─── MLP tile mask helpers ────────────────────────────────────────────────────

def apply_tile_mask(weight: jnp.ndarray, mask: jnp.ndarray, tile_size: int = 16) -> jnp.ndarray:
    out_f, in_f = weight.shape
    mask_expanded = jnp.repeat(jnp.repeat(mask, tile_size, axis=0), tile_size, axis=1)
    return weight * mask_expanded.astype(weight.dtype)


def compute_tile_scores(grad_weight: jnp.ndarray, tile_size: int = 16) -> jnp.ndarray:
    out_f, in_f = grad_weight.shape
    R = out_f // tile_size
    C = in_f  // tile_size
    blocked = grad_weight.reshape(R, tile_size, C, tile_size)
    return jnp.sqrt((blocked.astype(jnp.float32) ** 2).sum(axis=(1, 3)))


# ─── Multi-head causal attention ──────────────────────────────────────────────

class MultiHeadAttention(nn.Module):
    n_heads:     int
    d_model:     int
    max_seq_len: int = 1024

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        B, S, D = x.shape
        head_dim = self.d_model // self.n_heads

        qkv = nn.Dense(3 * self.d_model, use_bias=False, dtype=jnp.bfloat16,
                       name="qkv")(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)

        def reshape(t):
            return t.reshape(B, S, self.n_heads, head_dim).transpose(0, 2, 1, 3)

        q, k, v = reshape(q), reshape(k), reshape(v)

        freqs = precompute_rope_freqs(head_dim, self.max_seq_len)
        q = apply_rope(q.astype(jnp.float32), freqs).astype(jnp.bfloat16)
        k = apply_rope(k.astype(jnp.float32), freqs).astype(jnp.bfloat16)

        scale       = math.sqrt(head_dim)
        attn_logits = jnp.matmul(q, k.transpose(0, 1, 3, 2)) / scale
        causal_mask = jnp.tril(jnp.ones((S, S), dtype=jnp.bool_))[None, None]
        NEG_INF     = jnp.finfo(jnp.bfloat16).min
        attn_logits = jnp.where(causal_mask, attn_logits, NEG_INF)
        attn_w      = jax.nn.softmax(attn_logits.astype(jnp.float32), axis=-1).astype(jnp.bfloat16)

        out = jnp.matmul(attn_w, v).transpose(0, 2, 1, 3).reshape(B, S, D)
        return nn.Dense(self.d_model, use_bias=False, dtype=jnp.bfloat16,
                        name="out")(out)


# ─── MLP block (dense weights + external tile mask) ───────────────────────────

class MLPBlock(nn.Module):
    d_model:   int
    d_ff:      int
    tile_size: int = 16

    @nn.compact
    def __call__(
        self,
        x: jnp.ndarray,
        fc1_mask: Optional[jnp.ndarray] = None,
        fc2_mask: Optional[jnp.ndarray] = None,
    ) -> jnp.ndarray:
        w1 = self.param("fc1_w", nn.initializers.normal(0.02), (self.d_ff, self.d_model))
        b1 = self.param("fc1_b", nn.initializers.zeros, (self.d_ff,))
        w2 = self.param("fc2_w", nn.initializers.normal(0.02), (self.d_model, self.d_ff))
        b2 = self.param("fc2_b", nn.initializers.zeros, (self.d_model,))

        w1_eff = apply_tile_mask(w1, fc1_mask, self.tile_size) if fc1_mask is not None else w1
        w2_eff = apply_tile_mask(w2, fc2_mask, self.tile_size) if fc2_mask is not None else w2

        x16   = x.astype(jnp.bfloat16)
        w1_16 = w1_eff.astype(jnp.bfloat16)
        w2_16 = w2_eff.astype(jnp.bfloat16)

        hidden = jnp.dot(x16, w1_16.T) + b1.astype(jnp.bfloat16)
        hidden = jax.nn.gelu(hidden)
        out    = jnp.dot(hidden, w2_16.T) + b2.astype(jnp.bfloat16)
        return out


# ─── Transformer block (pre-norm) ─────────────────────────────────────────────

class TransformerBlock(nn.Module):
    d_model:     int
    n_heads:     int
    d_ff:        int
    max_seq_len: int  = 1024
    tile_size:   int  = 16

    @nn.compact
    def __call__(
        self,
        x: jnp.ndarray,
        fc1_mask: Optional[jnp.ndarray] = None,
        fc2_mask: Optional[jnp.ndarray] = None,
    ) -> jnp.ndarray:
        x = x + MultiHeadAttention(
            n_heads=self.n_heads,
            d_model=self.d_model,
            max_seq_len=self.max_seq_len,
            name="attn",
        )(RMSNorm(name="norm_attn")(x))
        x = x + MLPBlock(
            d_model=self.d_model,
            d_ff=self.d_ff,
            tile_size=self.tile_size,
            name="mlp",
        )(RMSNorm(name="norm_mlp")(x), fc1_mask, fc2_mask)
        return x


# ─── Diagonal injection ───────────────────────────────────────────────────────

class DiagonalInjection(nn.Module):
    d_model:    int
    init_decay: float = 0.447

    @nn.compact
    def __call__(self, h: jnp.ndarray, e: jnp.ndarray) -> jnp.ndarray:
        log_A = self.param("log_A", nn.initializers.zeros, (self.d_model,))

        target_dt      = -math.log(self.init_decay)
        init_log_dt    = math.log(math.exp(target_dt) - 1.0)
        log_dt = self.param("log_dt",
                            nn.initializers.constant(init_log_dt),
                            (self.d_model,))

        A     = -jnp.exp(log_A)
        dt    = jax.nn.softplus(log_dt)
        decay = jnp.exp(dt * A)
        return (decay * h + dt * e).astype(h.dtype)


# ─── Looped Transformer ───────────────────────────────────────────────────────

class LoopedTransformer(nn.Module):
    d_model:     int
    n_heads:     int
    d_ff:        int
    n_prelude:   int
    n_core:      int
    n_coda:      int
    vocab_size:  int
    max_seq_len: int = 1024
    tile_size:   int = 16
    max_depth:   int = 4

    @nn.compact
    def __call__(
        self,
        input_ids:  jnp.ndarray,
        depths:     jnp.ndarray,
        n_max:      jnp.ndarray,
        labels:     Optional[jnp.ndarray] = None,
        masks:      Optional[dict]         = None,
    ) -> dict:
        cfg = dict(
            d_model=self.d_model, n_heads=self.n_heads, d_ff=self.d_ff,
            max_seq_len=self.max_seq_len, tile_size=self.tile_size,
        )
        B, S = input_ids.shape

        def _get_masks(name):
            if masks is None:
                return None, None
            entry = masks.get(name, {})
            return entry.get("fc1"), entry.get("fc2")

        embed = nn.Embed(
            num_embeddings=self.vocab_size,
            features=self.d_model,
            embedding_init=nn.initializers.normal(0.02),
            name="embed",
        )
        x = embed(input_ids).astype(jnp.bfloat16)

        for i in range(self.n_prelude):
            m1, m2 = _get_masks(f"prelude_{i}")
            x = TransformerBlock(**cfg, name=f"prelude_{i}")(x, m1, m2)

        e = RMSNorm(name="input_norm")(x)

        # h must be random init — matches PyTorch's normal_(0, 0.02)
        h = jax.random.normal(self.make_rng("state"), e.shape, dtype=jnp.float32) * 0.02
        h = h.astype(e.dtype)

        injection   = DiagonalInjection(d_model=self.d_model, name="injection")
        core_blocks = [TransformerBlock(**cfg, name=f"core_{i}") for i in range(self.n_core)]
        core_masks = [_get_masks(f"core_{i}") for i in range(self.n_core)]

        for t in range(self.max_depth):
            h_new = injection(h, e)
            for blk, (m1, m2) in zip(core_blocks, core_masks):
                h_new = blk(h_new, m1, m2)

            h_sg = jax.lax.stop_gradient(h_new)
            is_nograd = jnp.array(t) < n_max
            h_new = jnp.where(is_nograd[:, None, None], h_sg, h_new)

            active = (jnp.array(t) < depths)[:, None, None]
            h = jnp.where(active, h_new, h)

        x = h
        for i in range(self.n_coda):
            m1, m2 = _get_masks(f"coda_{i}")
            x = TransformerBlock(**cfg, name=f"coda_{i}")(x, m1, m2)

        x = RMSNorm(name="final_norm")(x)
        logits = embed.attend(x)

        loss = None
        if labels is not None:
            loss = optax.softmax_cross_entropy_with_integer_labels(
                logits.reshape(-1, self.vocab_size),
                labels.reshape(-1),
            ).mean()

        return {"logits": logits, "loss": loss}


def make_model(d_model, n_heads, d_ff, n_core) -> LoopedTransformer:
    return LoopedTransformer(
        d_model=d_model, n_heads=n_heads, d_ff=d_ff,
        n_prelude=N_PRELUDE, n_core=n_core, n_coda=N_CODA,
        vocab_size=VOCAB_SIZE, max_seq_len=MAX_SEQ_LEN,
        tile_size=TILE_SIZE, max_depth=MAX_DEPTH,
    )


def count_params(params) -> int:
    return sum(p.size for p in jax.tree_util.tree_leaves(params))


print("Model classes defined. ✓")

## Cell 5 — CMS Scoring & Pruning

Pure-functional CMS (Continuum Memory System) implementation compatible with `jax.jit`.

**State**: per-layer `CMSState` holds accumulated gradient tile norms.

**Timing** (critical):
1. `accumulate_scores(cms, grad_w)` — called with raw gradients **before** optax update
2. `score_step(cms)` — normalise every `SCORE_INTERVAL` steps  
3. `prune_tiles(cms, mask, PRUNE_FRACTION)` — per-row bottom-k every `PRUNE_INTERVAL` steps
4. After pruning: re-zero optimizer momentum/variance for newly dead tiles

In [ ]:
from typing import Tuple


# ─── CMS state ────────────────────────────────────────────────────────────────

class CMSState(NamedTuple):
    """Per-MLP weight CMS state.  Carried as a plain pytree (NamedTuple).

    Fields
    ------
    scores      : [R, C] float32 — accumulated gradient Frobenius norms
    alive       : [R, C] bool    — True = tile is alive
    accum_count : scalar int32   — steps since last normalisation
    """
    scores:      jnp.ndarray   # [R, C]
    alive:       jnp.ndarray   # [R, C] bool
    accum_count: jnp.ndarray   # scalar


def init_cms(d_out: int, d_in: int, tile_size: int) -> CMSState:
    R = d_out // tile_size
    C = d_in  // tile_size
    return CMSState(
        scores=jnp.zeros((R, C), dtype=jnp.float32),
        alive=jnp.ones((R, C),  dtype=jnp.bool_),
        accum_count=jnp.array(0, dtype=jnp.int32),
    )


def accumulate_scores(cms: CMSState, grad_w: jnp.ndarray, tile_size: int = TILE_SIZE) -> CMSState:
    """Add per-tile gradient Frobenius norms to the score buffer.

    Call BEFORE optax apply_updates — mirroring the PyTorch contract of
    calling after loss.backward() and before optimizer.step().

    grad_w : [d_out, d_in]
    """
    tile_scores = compute_tile_scores(grad_w, tile_size)       # [R, C]
    tile_scores = jnp.where(cms.alive, tile_scores, 0.0)       # dead tiles stay 0
    return CMSState(
        scores=cms.scores + tile_scores,
        alive=cms.alive,
        accum_count=cms.accum_count + 1,
    )


def score_step(cms: CMSState) -> CMSState:
    """Normalise accumulated scores by step count. Call every SCORE_INTERVAL steps."""
    n = jnp.maximum(cms.accum_count.astype(jnp.float32), 1.0)
    return CMSState(
        scores=cms.scores / n,
        alive=cms.alive,
        accum_count=jnp.array(0, dtype=jnp.int32),
    )


def prune_tiles(cms: CMSState, fraction: float) -> CMSState:
    """Per-row: mark the bottom `fraction` of alive tiles as dead.

    Dead tiles get score 0 and alive=False. Col indices are not stored here;
    the caller reads cms.alive as the new mask for apply_tile_mask.
    """
    def _prune_row(scores_row: jnp.ndarray, alive_row: jnp.ndarray):
        C = scores_row.shape[0]
        n_alive  = alive_row.sum().astype(jnp.float32)
        n_to_kill = jnp.maximum(1, jnp.floor(n_alive * fraction).astype(jnp.int32))

        # Dead tiles masked to +inf so they're never selected for killing
        masked = jnp.where(alive_row, scores_row, jnp.inf)
        sorted_idx   = jnp.argsort(masked)               # ascending
        kill_pos     = jnp.arange(C) < n_to_kill         # first n_to_kill slots
        kill_at_orig = jnp.zeros(C, dtype=jnp.bool_).at[sorted_idx].set(kill_pos)
        kill_at_orig = kill_at_orig & alive_row           # only kill alive tiles
        new_alive    = alive_row & ~kill_at_orig
        return new_alive

    new_alive = jax.vmap(_prune_row)(cms.scores, cms.alive)  # [R, C]
    new_scores = jnp.where(new_alive, cms.scores, 0.0)
    return CMSState(scores=new_scores, alive=new_alive, accum_count=cms.accum_count)


def get_density(cms: CMSState) -> float:
    return float(cms.alive.mean())


# ─── CMS registry ─────────────────────────────────────────────────────────────
# Keys follow param path pattern: e.g. "prelude_0/mlp/fc1_w"

def _build_cms_registry(n_prelude: int, n_core: int, n_coda: int,
                         d_model: int, d_ff: int, tile_size: int) -> dict:
    """Create empty CMSState for every MLP weight in the model."""
    reg = {}
    block_names = (
        [f"prelude_{i}" for i in range(n_prelude)]
        + [f"core_{i}"   for i in range(n_core)]
        + [f"coda_{i}"   for i in range(n_coda)]
    )
    for name in block_names:
        # fc1: d_ff × d_model,  fc2: d_model × d_ff
        reg[f"{name}/mlp/fc1_w"] = init_cms(d_ff,    d_model, tile_size)
        reg[f"{name}/mlp/fc2_w"] = init_cms(d_model, d_ff,    tile_size)
    return reg


def cms_to_masks(cms_reg: dict) -> dict:
    """Convert CMS registry to the masks dict expected by LoopedTransformer.

    Returns: {block_name: {'fc1': [R,C] bool, 'fc2': [R,C] bool}}
    """
    masks = {}
    for key, state in cms_reg.items():
        # key = "prelude_0/mlp/fc1_w"
        parts    = key.split("/")
        blk_name = parts[0]               # e.g. "prelude_0"
        fc_key   = parts[-1].replace("_w", "")  # fc1 or fc2
        masks.setdefault(blk_name, {})[fc_key] = state.alive
    return masks


print("CMS scoring & pruning defined. ✓")

## Cell 6 — Training Loop (Grid Search)

`train_one(d_model, n_heads, d_ff, n_core)` runs a complete training experiment and returns
a results dict. The outer loop iterates over `GRID` and saves results incrementally.

Three phases logged separately in wandb:
- **DENSE** — full dense MLP (no masks)
- **PRUNE R{n}** — active pruning rounds
- **SPARSE** — training continues at final density after PRUNE_END

CMS accumulation happens after `jax.grad()` and before `optax.apply_updates()`.
Depth is sampled in Python (host) before each `jit`-ted train step.

Results saved to `scaling_results.json` after each config completes.

In [ ]:
import time
import json
import gc
import os
import pickle


import flax.serialization as serialization

CKPT_INTERVAL = 5000

def save_checkpoint(drive_dir, run_name, step, params, opt_state, cms_reg, prune_round, key):
    """Save full training state to Drive for mid-run resume."""
    ckpt_path = os.path.join(drive_dir, f"ckpt_{run_name}.pkl")
    import pickle
    ckpt = {
        "step": step,
        "params": serialization.to_bytes(params),
        "opt_state": serialization.to_bytes(opt_state),
        "cms_scores": {k: {"scores": np.array(v.scores), "alive": np.array(v.alive), "accum_count": int(v.accum_count)} for k, v in cms_reg.items()},
        "prune_round": prune_round,
        "key": np.array(key),
    }
    with open(ckpt_path, "wb") as f:
        pickle.dump(ckpt, f)
    print(f"  💾 Checkpoint saved: step {step} → {ckpt_path}")

def load_checkpoint(drive_dir, run_name, params_template, opt_state_template):
    """Load checkpoint if it exists. Returns (step, params, opt_state, cms_data, prune_round, key) or None."""
    import pickle
    ckpt_path = os.path.join(drive_dir, f"ckpt_{run_name}.pkl")
    if not os.path.exists(ckpt_path):
        return None
    with open(ckpt_path, "rb") as f:
        ckpt = pickle.load(f)
    params = serialization.from_bytes(params_template, ckpt["params"])
    opt_state = serialization.from_bytes(opt_state_template, ckpt["opt_state"])
    key = jnp.array(ckpt["key"])
    print(f"  📂 Checkpoint loaded: step {ckpt['step']} from {ckpt_path}")
    return ckpt["step"], params, opt_state, ckpt["cms_scores"], ckpt["prune_round"], key

def delete_checkpoint(drive_dir, run_name):
    """Clean up checkpoint after successful completion."""
    ckpt_path = os.path.join(drive_dir, f"ckpt_{run_name}.pkl")
    if os.path.exists(ckpt_path):
        os.remove(ckpt_path)

# ─── Depth sampling (outside jit) ────────────────────────────────────────────

def sample_depths(key: jax.Array, batch_size: int, mean_depth: int,
                  max_depth: int, bptt_depth: int, use_poisson: bool):
    if use_poisson:
        total = jax.random.poisson(key, lam=mean_depth, shape=(batch_size,))
        total = jnp.clip(total, 1, max_depth).astype(jnp.int32)
    else:
        total = jnp.full((batch_size,), mean_depth, dtype=jnp.int32)

    k_per_seq = jnp.minimum(total, bptt_depth)
    n_per_seq = total - k_per_seq
    return total, n_per_seq


# ─── Optimizer factory ────────────────────────────────────────────────────────

def make_optimizer():
    warmup  = optax.linear_schedule(0.0, LR, WARMUP_STEPS)
    decay   = optax.cosine_decay_schedule(LR, decay_steps=1_000_000, alpha=0.1)
    sched   = optax.join_schedules([warmup, decay], [WARMUP_STEPS])
    return optax.chain(
        optax.clip_by_global_norm(GRAD_CLIP),
        optax.adamw(sched, weight_decay=WEIGHT_DECAY, b1=0.9, b2=0.95),
    )


# ─── Opt state mask zeroing (after pruning) ──────────────────────────────────

def zero_opt_state_for_dead_tiles(
    opt_state, params, cms_reg: dict, tile_size: int = TILE_SIZE
):
    masks_flat = {}
    for cms_key, state in cms_reg.items():
        parts  = cms_key.split("/")
        blk    = parts[0]
        fc     = parts[-1]
        R, C   = state.alive.shape
        m_exp  = jnp.repeat(jnp.repeat(state.alive, tile_size, axis=0), tile_size, axis=1)
        masks_flat[(blk, fc)] = m_exp.astype(jnp.float32)

    def _mask_tree(tree):
        leaves, treedef = jax.tree_util.tree_flatten_with_path(tree)
        new_leaves = []
        for path, leaf in leaves:
            if hasattr(leaf, 'shape') and len(leaf.shape) == 2:
                path_str = "/".join(str(p) for p in path)
                for (blk, fc), mask in masks_flat.items():
                    if blk in path_str and fc in path_str and leaf.shape == mask.shape:
                        leaf = leaf * mask
                        break
            new_leaves.append(leaf)
        return jax.tree_util.tree_unflatten(treedef, new_leaves)

    return _mask_tree(opt_state)


# ─── Phase label ──────────────────────────────────────────────────────────────

def phase_label(step: int, prune_round: int) -> str:
    if step < PRUNE_START or N_PRUNE_ROUNDS == 0:
        return "DENSE"
    if PRUNE_START <= step < PRUNE_END:
        return f"PRUNE R{prune_round}"
    return "SPARSE"


# ─── Single training run ─────────────────────────────────────────────────────

def train_one(d_model, n_heads, d_ff, n_core) -> dict:
    effective_depth = N_PRELUDE + n_core * MEAN_DEPTH + N_CODA
    run_name        = f"d{d_model}_c{n_core}"

    key    = jax.random.PRNGKey(SEED)
    key, k_init, k_state = jax.random.split(key, 3)

    model = make_model(d_model, n_heads, d_ff, n_core)
    tx    = make_optimizer()

    dummy_ids    = jnp.ones((BATCH_SIZE, MAX_SEQ_LEN), dtype=jnp.int32)
    dummy_depths = jnp.full((BATCH_SIZE,), MEAN_DEPTH, dtype=jnp.int32)
    dummy_nmax   = jnp.zeros((BATCH_SIZE,), dtype=jnp.int32)

    params = model.init(
        {"params": k_init, "state": k_state},
        dummy_ids, dummy_depths, dummy_nmax,
    )["params"]
    opt_state = tx.init(params)
    n_params  = count_params(params)
    print(f"  Parameters: {n_params:,} ({n_params/1e6:.1f}M)")

    cms_reg = _build_cms_registry(
        N_PRELUDE, n_core, N_CODA, d_model, d_ff, TILE_SIZE
    )
    current_masks = None
    prune_round   = 0

    train_sampler.pos = 0
    train_sampler.epoch = 0
    start_step = 0

    # Try to resume from mid-run checkpoint
    ckpt = load_checkpoint(DRIVE_DIR, run_name, params, opt_state)
    if ckpt is not None:
        start_step, params, opt_state, cms_data, prune_round, key = ckpt
        for ck, cv in cms_data.items():
            if ck in cms_reg:
                cms_reg[ck] = CMSState(
                    scores=jnp.array(cv["scores"]),
                    alive=jnp.array(cv["alive"]),
                    accum_count=jnp.array(cv["accum_count"], dtype=jnp.int32),
                )
        current_masks = cms_to_masks(cms_reg) if start_step >= PRUNE_START else None
        # Advance sampler to match step position
        train_sampler.pos = (start_step * BATCH_SIZE * MAX_SEQ_LEN) % len(train_buf)
        train_sampler.epoch = (start_step * BATCH_SIZE * MAX_SEQ_LEN) // len(train_buf)
        print(f"  Resuming from step {start_step}, prune_round={prune_round}")

    wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        config={
            "d_model": d_model, "n_heads": n_heads, "d_ff": d_ff,
            "n_prelude": N_PRELUDE, "n_core": n_core, "n_coda": N_CODA,
            "mean_depth": MEAN_DEPTH, "max_depth": MAX_DEPTH,
            "bptt_depth": BPTT_DEPTH, "effective_depth": effective_depth,
            "n_params": n_params,
            "lr": LR, "total_steps": TOTAL_STEPS, "batch_size": BATCH_SIZE,
            "prune_fraction": PRUNE_FRACTION, "prune_interval": PRUNE_INTERVAL,
            "prune_start": PRUNE_START, "prune_end": PRUNE_END,
            "n_prune_rounds": N_PRUNE_ROUNDS, "final_density": FINAL_DENSITY,
            "tile_size": TILE_SIZE, "use_poisson": USE_POISSON,
        }
    )

    @jax.jit
    def train_step(params, x, y, depths, n_max, masks, rng_key):
        def loss_fn(p):
            out = model.apply(
                {"params": p}, x, depths, n_max, y, masks,
                rngs={"state": rng_key},
            )
            return out["loss"], out
        (loss, out), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
        return loss, grads

    @jax.jit
    def apply_grads(params, opt_state, grads):
        updates, new_opt_state = tx.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        return new_params, new_opt_state

    @jax.jit
    def eval_step(params, x, y, masks, rng_key):
        depths = jnp.full((x.shape[0],), MEAN_DEPTH, dtype=jnp.int32)
        n_max  = jnp.zeros((x.shape[0],), dtype=jnp.int32)
        out    = model.apply(
            {"params": params}, x, depths, n_max, y, masks,
            rngs={"state": rng_key},
        )
        return out["loss"]

    def evaluate(params, masks, n_batches: int = 20) -> float:
        losses = []
        nonlocal key
        for _ in range(n_batches):
            key, eval_key = jax.random.split(key)
            xv, yv = get_val_batch()
            losses.append(float(eval_step(params, xv, yv, masks, eval_key)))
        return sum(losses) / len(losses)

    t0         = time.time()
    step_times = []
    final_train_loss = float("nan")
    final_val_ppl    = float("nan")

    for step in range(start_step, TOTAL_STEPS):
        x, y = train_sampler.get_batch()

        key, depth_key, state_key = jax.random.split(key, 3)
        depths, n_max = sample_depths(
            depth_key, BATCH_SIZE, MEAN_DEPTH, MAX_DEPTH, BPTT_DEPTH, USE_POISSON
        )

        ts = time.time()
        loss, grads = train_step(params, x, y, depths, n_max, current_masks, state_key)
        loss.block_until_ready()
        step_times.append(time.time() - ts)

        if step <= PRUNE_END:
            for cms_key in list(cms_reg.keys()):
                parts    = cms_key.split("/")
                blk_name = parts[0]
                fc_name  = parts[-1]
                try:
                    grad_w = grads[blk_name]["mlp"][fc_name]
                    cms_reg[cms_key] = accumulate_scores(cms_reg[cms_key], grad_w, TILE_SIZE)
                except (KeyError, TypeError):
                    pass

        params, opt_state = apply_grads(params, opt_state, grads)

        if step > 0 and step % SCORE_INTERVAL == 0 and step <= PRUNE_END:
            for k in cms_reg:
                cms_reg[k] = score_step(cms_reg[k])

        if (PRUNE_START <= step < PRUNE_END
                and step > PRUNE_START
                and (step - PRUNE_START) % PRUNE_INTERVAL == 0):
            prune_round += 1
            for k in cms_reg:
                cms_reg[k] = prune_tiles(cms_reg[k], PRUNE_FRACTION)
            current_masks = cms_to_masks(cms_reg)
            opt_state = zero_opt_state_for_dead_tiles(opt_state, params, cms_reg)
            avg_density = sum(get_density(s) for s in cms_reg.values()) / len(cms_reg)
            print(f"  🔪 PRUNE R{prune_round} @ step {step} | density={avg_density:.1%}")

        if step % LOG_INTERVAL == 0:
            loss_val    = float(loss)
            ppl         = math.exp(min(loss_val, 20.0))
            avg_density = sum(get_density(s) for s in cms_reg.values()) / max(len(cms_reg), 1)
            window      = step_times[-LOG_INTERVAL:] if len(step_times) >= LOG_INTERVAL else step_times
            sps         = 1.0 / (sum(window) / len(window))
            phase       = phase_label(step, prune_round)
            eta_h       = (TOTAL_STEPS - step) / sps / 3600
            final_train_loss = loss_val

            print(f"  step {step:6d} [{phase:>12s}] | "
                  f"loss {loss_val:.4f} | ppl {ppl:8.2f} | "
                  f"density {avg_density:.2%} | "
                  f"{sps:.1f} step/s | eta {eta_h:.1f}h"
                  f" | epoch {train_sampler.epoch}")

            wandb.log({
                "train/loss":       loss_val,
                "train/ppl":        ppl,
                "train/phase":      phase,
                "train/epoch":      train_sampler.epoch,
                "mlp/density":      avg_density,
                "perf/step_s":      1.0 / sps,
                "perf/steps_per_s": sps,
            }, step=step)

        if step % EVAL_INTERVAL == 0 and step > 0:
            val_loss = evaluate(params, current_masks)
            val_ppl  = math.exp(min(val_loss, 20.0))
            final_val_ppl = val_ppl
            print(f"  📊 EVAL @ step {step} | val_loss {val_loss:.4f} | val_ppl {val_ppl:.2f}")
            wandb.log({"val/loss": val_loss, "val/ppl": val_ppl}, step=step)

        # Checkpoint to Drive every CKPT_INTERVAL steps
        if step > 0 and step % CKPT_INTERVAL == 0:
            save_checkpoint(DRIVE_DIR, run_name, step, params, opt_state, cms_reg, prune_round, key)

    val_loss      = evaluate(params, current_masks, n_batches=50)
    final_val_ppl = math.exp(min(val_loss, 20.0))
    avg_density   = sum(get_density(s) for s in cms_reg.values()) / max(len(cms_reg), 1)
    elapsed       = time.time() - t0

    print(f"\n  Training complete in {elapsed/3600:.1f}h | epochs: {train_sampler.epoch}")
    print(f"  Final val PPL: {final_val_ppl:.2f}  |  density: {avg_density:.1%}")

    wandb.summary.update({
        "final_val_ppl":   final_val_ppl,
        "final_density":   avg_density,
        "n_params":        n_params,
        "effective_depth": effective_depth,
        "epochs":          train_sampler.epoch,
    })
    wandb.finish()

    delete_checkpoint(DRIVE_DIR, run_name)
    del params, opt_state, grads, model, tx, cms_reg
    jax.clear_caches()
    gc.collect()

    return {
        "d_model":          d_model,
        "n_heads":          n_heads,
        "d_ff":             d_ff,
        "n_core":           n_core,
        "n_params":         n_params,
        "effective_depth":  effective_depth,
        "final_train_loss": final_train_loss,
        "final_val_ppl":    final_val_ppl,
        "final_density":    avg_density,
        "epochs":           train_sampler.epoch,
    }


# ─── Grid search loop with auto-resume ────────────────────────────────────────

RESULTS_FILE = os.path.join(DRIVE_DIR, "scaling_results.json")

# Load previous results if resuming
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        all_results = json.load(f)
    completed = {(r["d_model"], r["n_core"]) for r in all_results}
    print(f"Resuming: {len(all_results)}/{len(GRID)} configs already complete")
    for r in all_results:
        print(f"  ✓ d{r['d_model']}_c{r['n_core']} → PPL {r['final_val_ppl']:.2f}")
else:
    all_results = []
    completed = set()
    print("Starting fresh grid search")

remaining = [(d, nh, df, nc) for d, nh, df, nc in GRID if (d, nc) not in completed]
print(f"Remaining: {len(remaining)} configs\n")

for i, (d_model, n_heads, d_ff, n_core) in enumerate(remaining):
    grid_idx = next(j for j, (d, _, _, nc) in enumerate(GRID) if d == d_model and nc == n_core)
    print(f"\n{'='*70}")
    print(f"Grid {grid_idx+1}/{len(GRID)} (run {len(all_results)+1}): d={d_model}, n_core={n_core}  "
          f"(eff_depth={N_PRELUDE + n_core * MEAN_DEPTH + N_CODA})")
    print(f"{'='*70}")
    result = train_one(d_model, n_heads, d_ff, n_core)
    all_results.append(result)
    with open(RESULTS_FILE, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"  Saved ({len(all_results)}/{len(GRID)} complete)")

print(f"\n{'='*70}")
print(f"All {len(GRID)} configs complete!")
print(f"{'='*70}")

## Cell 7 — Results: Heatmap + Scatter

Loads `scaling_results.json` and produces:
1. **2D heatmap**: x=d_model, y=n_core, color=val_ppl
2. **Scatter**: n_params vs val_ppl, colored by n_core
3. **Full results table** sorted by val_ppl

In [ ]:
RESULTS_FILE = os.path.join(DRIVE_DIR, "scaling_results.json")
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd

# ─── Load results ─────────────────────────────────────────────────────────────
with open(RESULTS_FILE) as f:
    all_results = json.load(f)

df = pd.DataFrame(all_results)
print(f"Loaded {len(df)} completed configs.\n")

# ─── Results table sorted by val_ppl ─────────────────────────────────────────
df_sorted = df.sort_values("final_val_ppl").reset_index(drop=True)
print("Full results (sorted by val_ppl):")
print(df_sorted[[
    "d_model", "n_core", "n_params", "effective_depth",
    "final_train_loss", "final_val_ppl", "final_density"
]].to_string(index=False))

# ── Best config ───────────────────────────────────────────────────────────────
best = df_sorted.iloc[0]
print(f"\nBest: d={best['d_model']}, n_core={best['n_core']}  "
      f"→ val_ppl={best['final_val_ppl']:.2f}  "
      f"({best['n_params']/1e6:.1f}M params)")

# ─── Plot 1: 2D Heatmap — d_model × n_core → val_ppl ─────────────────────────
d_vals    = sorted(df["d_model"].unique())
nc_vals   = sorted(df["n_core"].unique())

# Build matrix (rows=n_core, cols=d_model) — NaN for missing configs
ppl_matrix = np.full((len(nc_vals), len(d_vals)), np.nan)
for _, row in df.iterrows():
    ri = nc_vals.index(row["n_core"])
    ci = d_vals.index(row["d_model"])
    ppl_matrix[ri, ci] = row["final_val_ppl"]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Looped Block-ELL Scaling Law Grid", fontsize=14, fontweight="bold")

# Heatmap
ax = axes[0]
vmin, vmax = np.nanmin(ppl_matrix), np.nanmax(ppl_matrix)
im = ax.imshow(ppl_matrix, aspect="auto", cmap="RdYlGn_r", vmin=vmin, vmax=vmax,
               origin="lower")
ax.set_xticks(range(len(d_vals)))
ax.set_xticklabels([str(d) for d in d_vals])
ax.set_yticks(range(len(nc_vals)))
ax.set_yticklabels([str(n) for n in nc_vals])
ax.set_xlabel("d_model", fontsize=12)
ax.set_ylabel("n_core", fontsize=12)
ax.set_title("Val PPL Heatmap (lower = better)", fontsize=12)
plt.colorbar(im, ax=ax, label="Val PPL")
# Annotate cells
for ri in range(len(nc_vals)):
    for ci in range(len(d_vals)):
        v = ppl_matrix[ri, ci]
        if not np.isnan(v):
            ax.text(ci, ri, f"{v:.1f}", ha="center", va="center",
                    fontsize=8, color="black" if 0.3 < (v - vmin) / (vmax - vmin + 1e-9) < 0.7 else "white")

# Scatter: n_params vs val_ppl, colored by n_core
ax = axes[1]
cmap  = plt.cm.viridis
nc_unique = sorted(df["n_core"].unique())
norm  = mcolors.BoundaryNorm(
    boundaries=[n - 0.5 for n in nc_unique] + [nc_unique[-1] + 0.5],
    ncolors=len(nc_unique)
)
sc = ax.scatter(
    df["n_params"] / 1e6,
    df["final_val_ppl"],
    c=df["n_core"],
    cmap="viridis",
    s=100, zorder=3, edgecolors="k", linewidths=0.5
)
plt.colorbar(sc, ax=ax, label="n_core", ticks=nc_unique)
# Annotate each point with d_model
for _, row in df.iterrows():
    ax.annotate(
        f"d{int(row['d_model'])}",
        (row["n_params"] / 1e6, row["final_val_ppl"]),
        fontsize=7, xytext=(4, 2), textcoords="offset points"
    )
ax.set_xlabel("Parameters (M)", fontsize=12)
ax.set_ylabel("Val PPL", fontsize=12)
ax.set_title("n_params vs Val PPL (color = n_core)", fontsize=12)
ax.grid(alpha=0.3)
ax.invert_yaxis()  # lower PPL = better, put it at top

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, "scaling_grid_results.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to scaling_grid_results.png")

## Cell 9 — Scaling Law Summary from wandb

Pulls all completed runs from `looped-blockell-scaling` and produces the
iso-compute scaling law scatter plot. Run names follow `d{d_model}_c{n_core}` convention.

Can be re-run at any time during the grid search to see partial results.

In [ ]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt

# ─── Pull all runs from the scaling project ────────────────────────────────────
api = wandb.Api()
# Run names: d{d_model}_c{n_core}
all_runs = api.runs(
    "looped-blockell-scaling",
    filters={"state": "finished"},
)

rows = []
for r in all_runs:
    cfg  = r.config
    summ = r.summary
    # Prefer summary field set at end of run; fall back to last logged val/ppl
    val_ppl = summ.get("final_val_ppl") or summ.get("val/ppl")
    if val_ppl is None:
        continue
    # Parse d_model / n_core from run name (d256_c4) if not in config
    d_model      = cfg.get("d_model")
    n_core       = cfg.get("n_core")
    if d_model is None or n_core is None:
        parts = r.name.split("_")
        d_model = int(parts[0][1:]) if parts[0].startswith("d") else None
        n_core  = int(parts[1][1:]) if len(parts) > 1 and parts[1].startswith("c") else None
    rows.append({
        "run_name":       r.name,
        "d_model":        d_model,
        "n_core":         n_core,
        "n_heads":        cfg.get("n_heads"),
        "d_ff":           cfg.get("d_ff"),
        "effective_depth":cfg.get("effective_depth"),
        "final_density":  cfg.get("final_density", 1.0),
        "n_params":       cfg.get("n_params", 0),
        "val_ppl":        round(float(val_ppl), 2),
        "run_url":        r.url,
    })

if not rows:
    print(f"No finished runs found in 'looped-blockell-scaling'. "
          f"Grid search may still be running.")
else:
    df_all = pd.DataFrame(rows).sort_values("val_ppl").reset_index(drop=True)
    print(f"Found {len(df_all)} completed runs.\n")
    print(df_all[["run_name", "d_model", "n_core", "n_params",
                  "effective_depth", "val_ppl"]].to_string(index=False))

    # ── Scatter: params vs PPL, coloured by n_core ────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f"Looped Block-ELL Scaling Laws — looped-blockell-scaling ({len(df_all)} runs)",
                 fontsize=13, fontweight="bold")

    ax = axes[0]
    sc = ax.scatter(
        df_all["n_params"] / 1e6,
        df_all["val_ppl"],
        c=df_all["n_core"].astype(float),
        cmap="viridis", s=90, zorder=3, edgecolors="k", linewidths=0.5
    )
    plt.colorbar(sc, ax=ax, label="n_core")
    for _, row in df_all.iterrows():
        ax.annotate(f"  {row['run_name'][:10]}",
                    (row["n_params"] / 1e6, row["val_ppl"]), fontsize=7)
    ax.set_xlabel("Parameters (M)")
    ax.set_ylabel("Val PPL")
    ax.set_title("Width Scaling (colored by n_core)")
    ax.grid(alpha=0.3)

    ax = axes[1]
    sc2 = ax.scatter(
        df_all["effective_depth"].astype(float),
        df_all["val_ppl"],
        c=df_all["n_params"].astype(float) / 1e6,
        cmap="plasma", s=90, zorder=3, edgecolors="k", linewidths=0.5
    )
    plt.colorbar(sc2, ax=ax, label="Params (M)")
    for _, row in df_all.iterrows():
        ax.annotate(f"  d{int(row['d_model'])}",
                    (row["effective_depth"], row["val_ppl"]), fontsize=7)
    ax.set_xlabel("Effective Depth (1 + n_core×T + 1)")
    ax.set_ylabel("Val PPL")
    ax.set_title("Depth Scaling (colored by params)")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("scaling_laws_wandb.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved scaling_laws_wandb.png")

    best = df_all.iloc[0]
    print(f"\nBest config: {best['run_name']}  →  val_ppl={best['val_ppl']:.2f}  "
          f"({best['n_params']/1e6:.1f}M params, eff_depth={best['effective_depth']})")